In [ ]:
# ================================================================
# HALLUVERSE-M3 — QWEN3:4B DIALOGUE SUMMARIZATION (RESUMABLE)
# ================================================================
#
# Same task and format as the Gemma3:1B run: Qwen reads a dialogue and
# writes its OWN one-sentence summary (English dialogue -> English
# summary, Nepali dialogue -> Nepali summary). Saved next to the
# reference summary for later hallucination grading.
#
# IMPORTANT — qwen3:4b has "thinking mode" on by default (it reasons
# out loud before answering, like you saw in your test run). For a
# batch job this is much slower AND pollutes the output with reasoning
# text. This script disables thinking via the API "think": false flag,
# and ALSO strips any leftover <think>...</think> block as a safety net
# in case the flag is ignored by your Ollama version.
#
# Source file: already-cleaned paired CSV (same one used for the
# Gemma run) — Hexagon-Dataset-HalluVerseM3_PAIRED.csv
# ================================================================

import os
import re
import time
from pathlib import Path

import pandas as pd
import requests


# ================================================================
# 1. SETTINGS
# ================================================================

FOLDER = Path(
    r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"
)

SOURCE_FILE = FOLDER / "Hexagon-Dataset-HalluVerseM3_PAIRED.csv"

OUTPUT_FILE = FOLDER / "HalluVerseM3_Qwen3_4B_Summaries.csv"

OLLAMA_URL = "http://localhost:11434/api/chat"   # NOTE: /api/generate ignores "think": false
                                                    # for Qwen3-family models (known Ollama bug,
                                                    # github.com/ollama/ollama/issues/14793).
                                                    # /api/chat is the endpoint that actually
                                                    # respects the think flag.
MODEL = "qwen3:4b"


# ================================================================
# 2. LOAD SOURCE (already paired — no filtering needed)
# ================================================================

if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found:\n{SOURCE_FILE}")

df = pd.read_csv(SOURCE_FILE)

required = [
    "ID", "Dialogue_EN", "Reference_Summary_EN", "Topic_EN",
    "Dialogue_NE", "Reference_Summary_NE", "Topic_NE",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing source columns: {missing}")

df["ID_STR"] = df["ID"].astype(str).str.strip()
df = df.drop_duplicates(subset=["ID_STR"]).reset_index(drop=True)

print("Paired rows in source:", len(df))
print("Expected evaluations (paired x 2 languages):", len(df) * 2)


# ================================================================
# 3. LOAD OR CREATE CHECKPOINT
# ================================================================

CHECKPOINT_COLS = [
    "ID", "Language", "Topic",
    "Dialogue", "Reference Summary",
    "Gemma Raw Output", "Gemma Summary",   # kept same col names as before for compatibility
]

if OUTPUT_FILE.exists():
    results = pd.read_csv(OUTPUT_FILE)
    missing_result = [c for c in CHECKPOINT_COLS if c not in results.columns]
    if missing_result:
        raise ValueError(f"Checkpoint is missing columns: {missing_result}")
    results = results.drop_duplicates(subset=["ID", "Language"], keep="last").reset_index(drop=True)
else:
    results = pd.DataFrame(columns=CHECKPOINT_COLS)
    results.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Existing saved evaluations:", len(results))
if len(results):
    print(results["Language"].value_counts().to_dict())


# ================================================================
# 4. EXPECTED PAIRS
# ================================================================

expected_pairs = {
    (item_id, language)
    for item_id in df["ID_STR"]
    for language in ["English", "Nepali"]
}


def get_pairs(data):
    return {
        (str(row["ID"]).strip(), str(row["Language"]).strip())
        for _, row in data.iterrows()
    }


# ================================================================
# 5. OLLAMA CHECK
# ================================================================

def wait_for_ollama():
    while True:
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=10)
            if r.status_code == 200:
                print("Ollama online.")
                return
        except Exception:
            pass
        print("Ollama unavailable. Retrying in 10 seconds...")
        time.sleep(10)


wait_for_ollama()


# ================================================================
# 6. SAFE SAVE
# ================================================================

def safe_save(data):
    temp_file = OUTPUT_FILE.with_suffix(OUTPUT_FILE.suffix + ".tmp")
    while True:
        try:
            data.to_csv(temp_file, index=False, encoding="utf-8-sig")
            os.replace(temp_file, OUTPUT_FILE)
            return
        except PermissionError:
            print("CSV is open/locked. Close it and retrying in 5 seconds...")
            time.sleep(5)


# ================================================================
# 7. STRIP THINKING BLOCK (safety net if "think": false is ignored)
# ================================================================

THINK_BLOCK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)

def strip_thinking(text):
    text = THINK_BLOCK.sub("", text)
    # Some builds print "Thinking...\n....\n...done thinking." without tags
    text = re.sub(r"^Thinking\.\.\..*?done thinking\.\s*", "", text, flags=re.DOTALL | re.IGNORECASE)
    return text.strip()


# ================================================================
# 8. QWEN REQUEST
# ================================================================

def ask_qwen(prompt):
    retry = 0
    wait = 3
    while True:
        try:
            r = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "messages": [{"role": "user", "content": prompt}],
                    "stream": False,
                    "keep_alive": "5m",
                    "think": False,          # /api/chat actually respects this (unlike /api/generate)
                    "options": {"temperature": 0},
                },
                timeout=(15, 600),
            )
            if r.status_code == 200:
                raw = r.json().get("message", {}).get("content", "").strip()
                return strip_thinking(raw)
            print("Ollama HTTP error:", r.status_code, r.text[:300])
        except requests.exceptions.ConnectionError:
            print("Ollama connection lost.")
        except requests.exceptions.Timeout:
            print("Ollama timeout.")
        except Exception as e:
            print("Ollama error:", e)

        retry += 1
        print(f"Retry {retry} in {wait}s...")
        time.sleep(wait)
        wait = min(wait * 2, 30)


# ================================================================
# 9. ONE TARGET
# ================================================================

def evaluate_target(item_id, language, row):

    if language == "English":
        dialogue = str(row["Dialogue_EN"])
        reference_summary = row.get("Reference_Summary_EN", "")
        topic = row.get("Topic_EN", "")

        prompt = f"""You are a dialogue summarization model.

Read the following dialogue and write a concise, ONE-sentence summary.
Only include information stated in the dialogue. Do not add names,
facts, or details that are not present in the dialogue.
Do not show your reasoning — output ONLY the final summary sentence.

Dialogue:
{dialogue}

Summary:"""

    else:
        dialogue = str(row["Dialogue_NE"])
        reference_summary = row.get("Reference_Summary_NE", "")
        topic = row.get("Topic_NE", "")

        prompt = f"""तपाईं संवाद सारांश बनाउने मोडेल हुनुहुन्छ।

तलको संवाद पढेर एक वाक्यमा छोटो सारांश लेख्नुहोस्।
संवादमा नभएका नाम, तथ्य वा विवरणहरू थप नगर्नुहोस् — संवादमा भएको
कुरा मात्र समावेश गर्नुहोस्।
आफ्नो सोचाइ नदेखाउनुहोस् — अन्तिम सारांश वाक्य मात्र लेख्नुहोस्।

संवाद:
{dialogue}

सारांश:"""

    raw_output = ask_qwen(prompt)
    summary = raw_output.strip()

    return {
        "ID": item_id,
        "Language": language,
        "Topic": topic,
        "Dialogue": dialogue,
        "Reference Summary": reference_summary,
        "Gemma Raw Output": raw_output,   # column name kept for schema compatibility
        "Gemma Summary": summary,
    }


# ================================================================
# 10. RESUME LOOP
# ================================================================

while True:

    results = pd.read_csv(OUTPUT_FILE)
    completed_pairs = get_pairs(results)

    missing_pairs = sorted(
        expected_pairs - completed_pairs,
        key=lambda x: (x[0], 0 if x[1] == "English" else 1),
    )

    if not missing_pairs:
        print("\n" + "=" * 70)
        print(f"ALL {len(expected_pairs)} EVALUATIONS COMPLETE")
        print("=" * 70)
        break

    item_id, language = missing_pairs[0]
    row = df[df["ID_STR"] == item_id].iloc[0]

    print(f"\nProcessing: {item_id} | {language}")

    new_row = evaluate_target(item_id, language, row)

    results = pd.read_csv(OUTPUT_FILE)
    current_pairs = get_pairs(results)
    if (item_id, language) in current_pairs:
        print(f"{item_id} | {language} already saved.")
        continue

    results = pd.concat([results, pd.DataFrame([new_row])], ignore_index=True)
    safe_save(results)

    verified = pd.read_csv(OUTPUT_FILE)
    verified_pairs = get_pairs(verified)
    if (item_id, language) not in verified_pairs:
        raise RuntimeError(f"Save verification failed: {item_id} | {language}")

    print(
        f"{item_id} | {language} | "
        f"Saved={len(verified_pairs)} | "
        f"Remaining={len(expected_pairs - verified_pairs)}"
    )


# ================================================================
# 11. FINAL AUDIT
# ================================================================

final_df = pd.read_csv(OUTPUT_FILE)
final_pairs = get_pairs(final_df)
missing_final = expected_pairs - final_pairs
duplicates = int(final_df.duplicated(subset=["ID", "Language"], keep=False).sum())

print("\n" + "=" * 70)
print("FINAL AUDIT")
print("=" * 70)
print("Expected:", len(expected_pairs))
print("Actual:", len(final_pairs))
print("Missing:", len(missing_final))
print("Duplicate rows:", duplicates)
print("\nLanguage counts:")
print(final_df["Language"].value_counts())

if len(missing_final) == 0 and duplicates == 0:
    print("\nCOMPLETE AND VERIFIED")
else:
    print("\nSTILL INCOMPLETE")
    print("First missing:", list(missing_final)[:30])

print("\nFile:")
print(OUTPUT_FILE)

Paired rows in source: 199
Expected evaluations (paired x 2 languages): 398
Existing saved evaluations: 0
Ollama online.

Processing: train_10022 | English
